# Módulo 8 · Práctica: clasificador de dígitos con regresión softmax

Construyes desde cero, solo con NumPy, un clasificador de dígitos 8×8. El modelo es una capa lineal seguida de softmax y entrenada con entropía cruzada. **Es exactamente la última capa de un LLM**: allí se clasifica sobre unos 100.000 tokens; aquí, sobre 10 dígitos.

**Cómo trabajar:** ejecuta en orden. Completa las celdas `# TU CÓDIGO` y ejecuta la `# Comprobación` que las sigue.

| Parte | Qué haces |
| --- | --- |
| 0 | Datos: cargar, ver y dividir en entrenamiento / validación / prueba |
| 1 | Calentamiento: regresión lineal con las ecuaciones normales |
| 2 | Sobreajuste en directo: polinomios y regularización L2 |
| 3 | Softmax, entropía cruzada y su gradiente |
| 4 | Entrenar el clasificador con minibatches |
| 5 | Validación cruzada para elegir λ |
| 6 | Evaluación final y conexión con los LLM |
| 7 | Retos |

**Datos:** si tienes `scikit-learn`, se usa `load_digits` (viene incluido en el paquete, no descarga nada). Si no, se generan dígitos sintéticos con plantillas 8×8, desplazamientos y ruido.

## Parte 0 · Datos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

PLANTILLAS = """
..####.. ...##... ..####.. ..####.. ....##.. .######. ..####.. .######. ..####.. ..####..
.#....#. ..###... .#....#. .#....#. ...###.. .#...... .#...... ......#. .#....#. .#....#.
.#....#. ...##... ......#. ......#. ..#.##.. .#####.. .#...... .....#.. .#....#. .#....#.
.#....#. ...##... .....#.. ...###.. .#..##.. ......#. .#####.. ....#... ..####.. ..#####.
.#....#. ...##... ...##... ......#. .######. ......#. .#....#. ...#.... .#....#. ......#.
.#....#. ...##... ..#..... .#....#. ....##.. .#....#. .#....#. ...#.... .#....#. ......#.
..####.. .######. .######. ..####.. ....##.. ..####.. ..####.. ...#.... ..####.. ..####..
........ ........ ........ ........ ........ ........ ........ ........ ........ ........
"""
filas = [l.split() for l in PLANTILLAS.strip().splitlines()]
T = np.array([[[c == "#" for c in filas[r][k]] for r in range(8)] for k in range(10)], float)

def digitos_sinteticos(n_por_clase=180, ruido=0.15, seed=0):
    # plantilla 8x8 + desplazamiento de ±1 píxel + contraste aleatorio + ruido
    r = np.random.default_rng(seed)
    X, y = [], []
    for k in range(10):
        for _ in range(n_por_clase):
            img = np.roll(T[k], (r.integers(-1, 2), r.integers(-1, 2)), axis=(0, 1))
            img = np.clip(img * r.uniform(0.6, 1.0) + r.normal(0, ruido, (8, 8)), 0, 1)
            X.append(img.ravel()); y.append(k)
    return np.array(X), np.array(y)

try:
    from sklearn.datasets import load_digits     # viene dentro del paquete: no descarga nada
    d = load_digits()
    X_all, y_all = d.data / 16.0, d.target          # píxeles 0..16 -> 0..1
    fuente = "sklearn load_digits (1.797 dígitos manuscritos reales)"
except ImportError:
    X_all, y_all = digitos_sinteticos()
    fuente = "dígitos sintéticos (plantillas + ruido)"
print(fuente, X_all.shape)

In [ ]:
fig, ax = plt.subplots(2, 10, figsize=(10, 2.2))
for k in range(10):
    for f in range(2):
        i = np.flatnonzero(y_all == k)[f]
        ax[f, k].imshow(X_all[i].reshape(8, 8), cmap="gray_r"); ax[f, k].axis("off")
    ax[0, k].set_title(k)
plt.show()

Dividimos 60 % / 20 % / 20 %. El conjunto de **prueba** no se toca hasta la Parte 6: es la única estimación honesta del error de generalización.

In [ ]:
rng = np.random.default_rng(0)
idx = rng.permutation(len(y_all))
n20 = round(0.2 * len(y_all))
i_te, i_va, i_tr = idx[:n20], idx[n20:2 * n20], idx[2 * n20:]
X_tr, y_tr = X_all[i_tr], y_all[i_tr]
X_va, y_va = X_all[i_va], y_all[i_va]
X_te, y_te = X_all[i_te], y_all[i_te]
n, d = X_tr.shape
K = 10
print(f"entrenamiento {X_tr.shape}, validación {X_va.shape}, prueba {X_te.shape}")

## Parte 1 · Calentamiento: regresión lineal

Con la matriz de diseño X (una columna de unos para el sesgo) los mínimos cuadrados se resuelven con las **ecuaciones normales**:

$$X^\top X\,\hat\theta = X^\top y \quad\Longrightarrow\quad \hat\theta = (X^\top X)^{-1}X^\top y$$

**Ejercicio 1.** Escribe `minimos_cuadrados(X, y)`. Usa `np.linalg.solve` (más estable y barato que invertir la matriz). Prueba con el ejemplo del texto: x = (0, 1, 2, 3), y = (1, 2, 4, 5).

In [ ]:
def minimos_cuadrados(X, y):
    # TU CÓDIGO
    return None

x4 = np.array([0., 1., 2., 3.]); y4 = np.array([1., 2., 4., 5.])
X4 = np.column_stack([np.ones(4), x4])
theta4 = minimos_cuadrados(X4, y4)
print(theta4)

In [ ]:
# Comprobación
assert theta4 is not None, "minimos_cuadrados devuelve None"
assert np.allclose(theta4, [0.9, 1.4]), f"Debe salir b = 0,9 y w = 1,4; sale {theta4}"
r = y4 - X4 @ theta4
assert np.allclose(X4.T @ r, 0), "El residuo debe ser ortogonal a las columnas de X"
print("Correcto: SSE =", round(r @ r, 4))

## Parte 2 · Sobreajuste en directo

Ajustamos polinomios de grado p a 10 puntos de y = sin(2πx) + ruido. Con p = 9 hay tantos parámetros como datos: el polinomio pasa por todos los puntos y generaliza fatal.

**Ejercicio 2.** Completa `ridge(X, y, lam)`, que resuelve $(X^\top X + \lambda I)\theta = X^\top y$. (Para simplificar, aquí también penalizamos el sesgo.)

In [ ]:
def ridge(X, y, lam):
    # TU CÓDIGO
    return None

In [ ]:
# Comprobación
assert ridge(X4, y4, 0.0) is not None, "ridge devuelve None"
assert np.allclose(ridge(X4, y4, 0.0), [0.9, 1.4]), "Con λ = 0 debe coincidir con mínimos cuadrados"
assert np.linalg.norm(ridge(X4, y4, 10.0)) < np.linalg.norm(ridge(X4, y4, 0.0)), "λ grande debe encoger θ"
print("Correcto")

In [ ]:
r2 = np.random.default_rng(1)
f = lambda x: np.sin(2 * np.pi * x)
xp_tr = np.sort(r2.uniform(0, 1, 10)); yp_tr = f(xp_tr) + r2.normal(0, 0.2, 10)
xp_va = r2.uniform(0, 1, 200);         yp_va = f(xp_va) + r2.normal(0, 0.2, 200)
poli = lambda x, p: np.vander(x, p + 1, increasing=True)   # columnas 1, x, x², ..., x^p
mse = lambda X, y, th: np.mean((X @ th - y) ** 2)

print("grado  λ       MSE ent.  MSE val.")
for p, lam in [(1, 0), (3, 0), (9, 0), (9, 1e-3)]:
    th = ridge(poli(xp_tr, p), yp_tr, lam) if lam else np.linalg.lstsq(poli(xp_tr, p), yp_tr, rcond=None)[0]
    print(f"{p:5d}  {lam:<6}  {mse(poli(xp_tr, p), yp_tr, th):8.4f}  {mse(poli(xp_va, p), yp_va, th):8.4f}")
    xx = np.linspace(0, 1, 200); plt.plot(xx, poli(xx, p) @ th, label=f"p={p}, λ={lam}")
plt.scatter(xp_tr, yp_tr, c="k", zorder=3); plt.ylim(-2, 2); plt.legend(); plt.show()

Fíjate: el error de entrenamiento baja siempre con el grado; el de validación baja y luego **sube**. Con p = 9 y un λ pequeño el error de validación vuelve a ser razonable: la regularización recorta la varianza.

## Parte 3 · Softmax, entropía cruzada y gradiente

Para un lote de n ejemplos, logits $Z = XW + b$ (forma n × K), probabilidades $P = \mathrm{softmax}(Z)$ por filas y etiquetas en *one-hot* $Y$:

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n} \log P_{i, y_i} + \frac{\lambda}{2}\lVert W\rVert^2, \qquad \nabla_W\mathcal{L} = \frac{1}{n}X^\top (P - Y) + \lambda W, \qquad \nabla_b\mathcal{L} = \frac{1}{n}\sum_i (P - Y)_i$$

**Ejercicio 3.** `softmax(Z)` estable por filas (resta el máximo de cada fila).  
**Ejercicio 4.** `perdida_y_gradiente(W, b, X, y, lam)` que devuelve `(L, gW, gb)`.

In [ ]:
def softmax(Z):
    # TU CÓDIGO
    return None

def one_hot(y, K=10):
    Y = np.zeros((len(y), K)); Y[np.arange(len(y)), y] = 1
    return Y

In [ ]:
# Comprobación
Zp = np.array([[2., 1., 0.], [1000., 1000., 1000.]])
Pp = softmax(Zp)
assert Pp is not None, "softmax devuelve None"
assert np.allclose(Pp[0], [0.6652, 0.2447, 0.0900], atol=1e-4), "softmax(2, 1, 0) debe ser (0,665; 0,245; 0,090)"
assert np.allclose(Pp[1], 1 / 3), "No es estable: con logits 1000 aparecen nan o inf"
print("Correcto")

In [ ]:
def perdida_y_gradiente(W, b, X, y, lam=0.0):
    # TU CÓDIGO
    L, gW, gb = None, None, None
    return L, gW, gb

In [ ]:
# Comprobación 1: con W = 0 el modelo es uniforme y la pérdida vale ln 10
L0, gW0, gb0 = perdida_y_gradiente(np.zeros((d, K)), np.zeros(K), X_tr, y_tr)
assert L0 is not None, "perdida_y_gradiente devuelve None"
assert np.isclose(L0, np.log(10)), f"Con W = 0 la pérdida debe ser ln 10 = 2,3026; sale {L0}"
assert gW0.shape == (d, K) and gb0.shape == (K,), "Formas del gradiente incorrectas"

# Comprobación 2: gradiente numérico por diferencias centradas (módulo 4)
rc = np.random.default_rng(3)
Wc, bc = rc.normal(0, 0.1, (d, K)), rc.normal(0, 0.1, K)
Xc, yc = X_tr[:20], y_tr[:20]
_, gW, gb = perdida_y_gradiente(Wc, bc, Xc, yc, lam=0.1)
h = 1e-5
for (i, j) in [(10, 3), (40, 7), (63, 0)]:
    E = np.zeros_like(Wc); E[i, j] = h
    num = (perdida_y_gradiente(Wc + E, bc, Xc, yc, 0.1)[0] - perdida_y_gradiente(Wc - E, bc, Xc, yc, 0.1)[0]) / (2 * h)
    assert abs(num - gW[i, j]) < 1e-6, f"gW[{i},{j}] analítico {gW[i, j]:.6f} ≠ numérico {num:.6f}"
e = np.zeros(K); e[5] = h
num = (perdida_y_gradiente(Wc, bc + e, Xc, yc, 0.1)[0] - perdida_y_gradiente(Wc, bc - e, Xc, yc, 0.1)[0]) / (2 * h)
assert abs(num - gb[5]) < 1e-6, "gb no coincide con el gradiente numérico"
print("Correcto: el gradiente analítico coincide con el numérico")

## Parte 4 · Entrenar con minibatches

Usamos SGD con minibatches (módulo 7). Cada época baraja los datos y recorre lotes de 64.

**Ejercicio 5.** Completa la actualización dentro del bucle: calcula el gradiente del lote y da un paso `W -= lr * gW`, `b -= lr * gb`.

In [ ]:
def entrenar(X, y, lam=0.0, lr=0.5, epocas=60, lote=64, seed=0):
    r = np.random.default_rng(seed)
    W, b = np.zeros((X.shape[1], K)), np.zeros(K)
    historia = []
    for ep in range(epocas):
        perm = r.permutation(len(y))
        for s in range(0, len(y), lote):
            j = perm[s:s + lote]
            # TU CÓDIGO: gradiente del lote X[j], y[j] y paso de descenso
            pass
        historia.append(perdida_y_gradiente(W, b, X, y, lam)[0])
    return W, b, historia

def exactitud(W, b, X, y):
    return np.mean((X @ W + b).argmax(axis=1) == y)

In [ ]:
W, b, hist = entrenar(X_tr, y_tr, lam=1e-3)
plt.plot(hist); plt.axhline(np.log(10), ls="--", c="gray", label="ln 10 (azar)")
plt.xlabel("época"); plt.ylabel("entropía cruzada (nats)"); plt.legend(); plt.show()
acc_tr, acc_va = exactitud(W, b, X_tr, y_tr), exactitud(W, b, X_va, y_va)
print(f"exactitud entrenamiento {acc_tr:.3f} · validación {acc_va:.3f}")

In [ ]:
# Comprobación
assert hist[-1] < 0.5 * np.log(10), "La pérdida apenas ha bajado: ¿actualizas W y b?"
assert acc_va > 0.85, f"La exactitud de validación debería superar 0,85; sale {acc_va:.3f}"
print("Correcto")

## Parte 5 · Validación cruzada para elegir λ

Con k = 5 *folds*: partes el entrenamiento en 5 trozos, entrenas con 4 y validas con el quinto, y rotas. La media de las 5 exactitudes estima el rendimiento de cada λ con menos varianza que una sola división.

**Ejercicio 6.** Completa `validacion_cruzada(X, y, lam, k)` usando `np.array_split` sobre una permutación fija de índices.

In [ ]:
def validacion_cruzada(X, y, lam, k=5, epocas=25):
    folds = np.array_split(np.random.default_rng(42).permutation(len(y)), k)
    accs = []
    for f in range(k):
        # TU CÓDIGO: va = folds[f]; tr = el resto concatenado; entrena y mide exactitud en va
        pass
    return np.mean(accs)

In [ ]:
lams = [0.0, 1e-3, 1e-2, 1e-1]
cv = {lam: validacion_cruzada(X_tr, y_tr, lam) for lam in lams}
for lam, a in cv.items():
    print(f"λ = {lam:<6} exactitud CV = {a:.3f}")
lam_mejor = max(cv, key=cv.get)
print("λ elegido:", lam_mejor)

In [ ]:
# Comprobación
assert all(np.isfinite(a) for a in cv.values()), "La validación cruzada devuelve nan: ¿rellenas accs?"
assert cv[1e-1] < cv[lam_mejor], "λ = 0,1 debería quedar por debajo del mejor (exceso de sesgo)"
assert cv[lam_mejor] > 0.85, "La mejor exactitud CV debería superar 0,85"
print("Correcto")

## Parte 6 · Evaluación final y conexión con los LLM

Reentrenamos con todo el entrenamiento y el λ elegido, y medimos **una sola vez** en prueba.

In [ ]:
W, b, _ = entrenar(X_tr, y_tr, lam=lam_mejor)
acc_te = exactitud(W, b, X_te, y_te)
P_te = softmax(X_te @ W + b)
L_te = -np.mean(np.log(P_te[np.arange(len(y_te)), y_te]))
print(f"prueba: exactitud {acc_te:.3f} · entropía cruzada {L_te:.3f} nats · perplejidad {np.exp(L_te):.2f}")

C = np.zeros((K, K), int)
np.add.at(C, (y_te, P_te.argmax(axis=1)), 1)
print("matriz de confusión (fila = real, columna = predicho)\n", C)

Cada columna de W es una **plantilla** de 64 pesos: el logit de la clase k es el producto escalar de la imagen con su plantilla (módulo 2). Visualízalas:

In [ ]:
fig, ax = plt.subplots(1, 10, figsize=(10, 1.4))
for k in range(10):
    ax[k].imshow(W[:, k].reshape(8, 8), cmap="RdBu_r"); ax[k].set_title(k); ax[k].axis("off")
plt.show()

**Ejercicio 7 (conexión con los LLM).** La última capa de un LLM es esta misma regresión softmax: entrada de dimensión d_model, salida de dimensión |V|. Calcula `params_digitos` (W y b de tu modelo) y `params_cabeza` para d_model = 4.096 y |V| = 32.000, **sin sesgo** (los LLM tipo Llama no lo usan en esta capa).

In [ ]:
params_digitos = None   # TU CÓDIGO
params_cabeza = None    # TU CÓDIGO

In [ ]:
# Comprobación
assert params_digitos == 650, "64·10 + 10 = 650"
assert params_cabeza == 131_072_000, "4.096 · 32.000 = 131.072.000"
print("Correcto: la cabeza del LLM tiene", f"{params_cabeza // params_digitos:,}".replace(",", "."), "veces más parámetros que tu clasificador")

## Parte 7 · Retos

1. **Temperatura.** Divide los logits de prueba entre T ∈ {0,5; 1; 2}. ¿Cambia la entropía cruzada? ¿Y la exactitud? ¿Por qué?
2. **Pocos datos.** Entrena con 50 ejemplos y λ ∈ {0; 0,01; 0,1}: mide entrenamiento frente a validación.
3. **Curva de aprendizaje.** Exactitud de validación frente a n = 50…1.000. Si se estanca, el límite es el **sesgo** del modelo lineal (el MLP del módulo 9 lo rompe).
4. **Mínimos cuadrados para clasificar.** Ajusta W con `ridge` a las etiquetas one-hot y compara con la softmax.

---
### Soluciones de referencia
Inténtalo antes de mirar.

In [ ]:
# Ejercicio 1
# def minimos_cuadrados(X, y):
#     return np.linalg.solve(X.T @ X, X.T @ y)
# Ejercicio 2
# def ridge(X, y, lam):
#     return np.linalg.solve(X.T @ X + lam * np.eye(X.shape[1]), X.T @ y)
# Ejercicio 3
# def softmax(Z):
#     Z = Z - Z.max(axis=1, keepdims=True); E = np.exp(Z)
#     return E / E.sum(axis=1, keepdims=True)
# Ejercicio 4
# def perdida_y_gradiente(W, b, X, y, lam=0.0):
#     m = len(y); P = softmax(X @ W + b)
#     L = -np.mean(np.log(P[np.arange(m), y] + 1e-12)) + lam / 2 * np.sum(W * W)
#     G = (P - one_hot(y, W.shape[1])) / m
#     return L, X.T @ G + lam * W, G.sum(axis=0)
# Ejercicio 5 (dentro del bucle de lotes)
#     _, gW, gb = perdida_y_gradiente(W, b, X[j], y[j], lam)
#     W -= lr * gW;  b -= lr * gb
# Ejercicio 6 (dentro del bucle de folds)
#     va = folds[f];  tr = np.concatenate([folds[g] for g in range(k) if g != f])
#     Wf, bf, _ = entrenar(X[tr], y[tr], lam=lam, epocas=epocas)
#     accs.append(exactitud(Wf, bf, X[va], y[va]))
# Ejercicio 7
# params_digitos = W.size + b.size        # 650
# params_cabeza = 4096 * 32000            # 131.072.000